# pandas: clean, group, and join data

**Stage:** Applied Python · **Suggested time:** 45–90 minutes (capstone: several sessions)

**Prerequisites:** Concepts from notebooks 00–19. No variables or files from other notebooks are required.

## What you will learn

- Inspect a DataFrame and its dtypes
- Handle missing and invalid values explicitly
- Aggregate and validate joins


## Your learning loop

1. Read the explanation and predict the next output.
2. Run the example with **Shift+Enter**.
3. Follow **Try a change**; explain the result in your own words.
4. Complete both exercises and enable their checks.
5. Build the mini-project, then use **Restart Kernel and Run All**.

Examples run as provided. Exercise starters are deliberately unfinished; their checks are disabled until you enable them. A successful Run All with checks disabled does **not** mean the exercises are complete.

Save a copy before extensive experiments. Use one notebook at a time to avoid mixing kernel state.


## 1. Load a small messy dataset

A DataFrame is a labeled table; each column is a Series. Inspect shape, dtypes, and missing values before calculating. This fixture is embedded so the notebook runs without downloading data.


In [1]:
import pandas as pd
from io import StringIO

raw_csv = "date,category,amount\n2025-01-01, Food ,120\n2025-01-02,travel,80\n2025-01-03,food,50\n2025-01-04,books,\n2025-01-05,travel,oops\n"
raw = pd.read_csv(StringIO(raw_csv))
print(raw)
print(raw.dtypes)
print("Missing per column:\n", raw.isna().sum())


         date category amount
0  2025-01-01    Food     120
1  2025-01-02   travel     80
2  2025-01-03     food     50
3  2025-01-04    books    NaN
4  2025-01-05   travel   oops
date        str
category    str
amount      str
dtype: object
Missing per column:
 date        0
category    0
amount      1
dtype: int64


**Try a change:** Add a malformed date to the fixture and inspect it before cleaning.


## 2. Clean with an explicit policy

Normalize category text, parse dates, and coerce invalid amounts to missing. Here we quarantine rows with missing, invalid, or negative amounts, rather than silently turning them into zero. Choose a policy that matches the real meaning of your data.


In [2]:
cleaned = raw.copy()
cleaned["category"] = cleaned["category"].str.strip().str.lower()
cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
cleaned["amount"] = pd.to_numeric(cleaned["amount"], errors="coerce")
valid = cleaned["date"].notna() & cleaned["category"].notna() & cleaned["category"].ne("") & cleaned["amount"].notna() & cleaned["amount"].ge(0)
rejected = cleaned.loc[~valid].copy()
accepted = cleaned.loc[valid].copy()
print("Accepted:\n", accepted)
print("Rejected:\n", rejected)
assert len(accepted) == 3 and len(rejected) == 2


Accepted:
         date category  amount
0 2025-01-01     food   120.0
1 2025-01-02   travel    80.0
2 2025-01-03     food    50.0
Rejected:
         date category  amount
3 2025-01-04    books     NaN
4 2025-01-05   travel     NaN


**Try a change:** Explain why filling a missing expense with zero might bias an analysis.


## 3. Aggregate and join

groupby splits rows into groups and aggregates them. A merge joins tables by keys. validate='many_to_one' rejects duplicate keys on the budget side, preventing unexpected multiplication of expense rows.


In [3]:
totals = accepted.groupby("category", as_index=False)["amount"].sum()
budgets = pd.DataFrame({"category": ["food", "travel"], "budget": [200, 100]})
report = totals.merge(budgets, on="category", how="left", validate="many_to_one")
report["remaining"] = report["budget"] - report["amount"]
print(report)
assert report.set_index("category").loc["food", "remaining"] == 30


  category  amount  budget  remaining
0     food   170.0     200       30.0
1   travel    80.0     100       20.0


**Try a change:** Add a duplicate food budget and observe the merge validation failure, then restore the data.


## Practice: solve before looking at the answers

Replace the starter code with your implementation. Run it, then change that exercise's `exercise_N_ready` flag to `True` and run its check cell. A failed assertion is feedback: compare the expected value with your result. Add at least one test of your own.


### Exercise 1: Top categories

Implement category_totals(frame): return a Series summing amount by category, sorted by category name.

<details>
<summary>Show a hint</summary>

Group by category, select amount, sum, then sort_index().

</details>


In [4]:
def category_totals(frame):
    pass


In [5]:
exercise_1_ready = False  # Set True when your implementation is ready.
if exercise_1_ready:
    sample = pd.DataFrame({"category": ["b", "a", "b"], "amount": [2, 3, 4]})
    assert category_totals(sample).to_dict() == {"a": 3, "b": 6}
    assert list(category_totals(sample).index) == ["a", "b"]
    print("Exercise 1: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.")


Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.


### Exercise 2: Select expensive rows

Return a copy containing rows with amount >= minimum, preserving row order and all columns.

<details>
<summary>Show a hint</summary>

Use .loc[mask].copy().

</details>


In [6]:
def select_expensive(frame, minimum):
    pass


In [7]:
exercise_2_ready = False  # Set True when your implementation is ready.
if exercise_2_ready:
    sample = pd.DataFrame({"amount": [10, 50, 100], "label": ["a", "b", "c"]})
    selected = select_expensive(sample, 50)
    assert selected["label"].tolist() == ["b", "c"]
    selected.loc[1, "amount"] = 999
    assert sample.loc[1, "amount"] == 50
    print("Exercise 2: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.")


Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.


## Build it yourself

Analyze a month of invented expenses. Produce a rejected-row table, category totals, and weekly totals. Check that category totals sum to the accepted grand total, and write three findings supported by values.

This open-ended extension has no single supplied solution. Record your design choices and test your result.


In [8]:
# Your mini-project workspace. Add more cells as needed.


## Common mistakes and limits

Avoid chained assignment; use .loc for explicit updates. Missing is not the same as zero. Floating-point currency is used here for teaching; use integer minor units or a suitable decimal approach for exact accounting.


## Before you move on

- [ ] I can explain the examples without reading the code.
- [ ] I changed at least two inputs and predicted the results.
- [ ] I completed both exercises and enabled their checks.
- [ ] I added a boundary or failure case of my own.
- [ ] I attempted the mini-project and recorded one lesson learned.

**Reflection:** What surprised you? What would you try next? Edit this Markdown cell with your notes.


[Course guide](../README.md) · [Worked solutions](../solutions/20_pandas_data_analysis_solutions.ipynb) · [Previous notebook](19_numpy_numerical_python.ipynb) · [Next notebook](21_matplotlib_visualization.ipynb)
